# Notebook 06 — Module 3: semantic intent verification

**ClinicalShield v2 · Revision pipeline**

## Purpose
Check whether a retrieved document actually answers the clinical query it was retrieved for,
and flag it when it does not.

## What v1 left unstated

**The threshold.** v1 used `tau = 0.3` with no justification given. Here the sweep runs on the
**validation** partition, the operating point is fixed, and test is evaluated once at that
point. A threshold chosen on the data it is then reported against is not a measurement.

**The markers.** v1 referred to "known override markers" and gave two examples. The full list
is in the config and exported with the module, so the claim is inspectable.

**The queries.** v1's Module 3 compares document against query, but the paper never says
where the query came from. Here each query is built from the host document's own metadata —
its drug and label section — so a document about warfarin interactions is matched against
*"What drug interactions are known for warfarin?"*

That construction matters for a reason worth stating: **the query is never derived from the
payload.** Building it from injected content would leak the label into the model's input and
inflate every number that follows.

## What this notebook is likely to show

v1 reported that Module 3 caught 33% of MPIB's V2 samples, all through the marker check
rather than similarity, with benign and adversarial cosine distributions overlapping almost
completely. If that holds here, the honest conclusion is that embedding distance contributes
little and the markers do the work — which is worth reporting plainly, because a module that
appears to work while one of its two signals does nothing is exactly what a careful reviewer
looks for.

Cell 6 separates the two signals so the contribution of each is visible rather than pooled.

## Reviewer points addressed
- **R3-Q1** what each layer contributes, measured per signal
- **R1-c** every rate reported with its n and its operating point
- **R4-8** `module3.py` exported to `src/` — Reviewer 4 found Module 3 missing entirely

## Runtime
GPU helpful but not required; roughly 5–15 minutes for the embeddings.

In [ ]:
# ── Cell 1 · Configuration ────────────────────────────────────────────────────
CONFIG = {
    "SEED": 42,

    # Module 3 compares a retrieved document against the clinical query.
    # v1 used tau = 0.3 with no justification. NB11 sweeps it; here the operating
    # point is chosen on the VALIDATION partition and then applied unchanged to
    # test, so the reported figure is not fitted to the data it is measured on.
    "TAU_INTENT": 0.30,
    "TAU_SWEEP": [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90],

    "EMBED_MODEL": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
    "MAX_LEN": 512,
    "BATCH_SIZE": 32,

    # Override markers. v1 called these "injection markers" without listing them.
    "OVERRIDE_MARKERS": [
        "ignore previous", "ignore all previous", "ignore the above",
        "disregard the", "disregard all", "system override", "new instruction",
        "override:", "admin:", "[admin]", "[config]", "[system]",
        "forget everything", "forget the", "prior context is void",
        "end of document", "attention model", "do not follow",
        "your reply must", "you must instead", "instead output",
    ],

    "USE_DRIVE":      True,
    "DRIVE_DIR":      "/content/drive/MyDrive/ClinicalShield_v2",
    "PUSH_TO_GITHUB": True,
    "GITHUB_REPO":    "NehlTech/ClinicalShield",
    "GITHUB_BRANCH":  "v2-revision",
    "GIT_USER_NAME":  "Adu-Boahene Bright",
    "GIT_USER_EMAIL": "baduboahene@st.knust.edu.gh",
}
SEED = CONFIG["SEED"]
print("tau (from v1)   :", CONFIG["TAU_INTENT"])
print("markers         :", len(CONFIG["OVERRIDE_MARKERS"]))
print("operating point chosen on validation, reported on test")


In [ ]:
# ── Cell 2 · Environment and inputs ───────────────────────────────────────────
import sys, os, json, random, hashlib, subprocess, shutil, time, math
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

random.seed(SEED); np.random.seed(SEED)

IN_COLAB = "google.colab" in sys.modules
DRIVE_ROOT, REPO_DIR = None, None

if IN_COLAB:
    if CONFIG["USE_DRIVE"]:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = Path(CONFIG["DRIVE_DIR"]); DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        print("drive :", DRIVE_ROOT)
    repo_name = CONFIG["GITHUB_REPO"].split("/")[-1]
    REPO_DIR = Path("/content") / repo_name
    if not REPO_DIR.exists():
        try:
            from google.colab import userdata
            tok = userdata.get("GITHUB_TOKEN")
            r = subprocess.run(["git","clone","-q",
                "https://" + tok + "@github.com/" + CONFIG["GITHUB_REPO"] + ".git",
                str(REPO_DIR)], capture_output=True, text=True)
            print("clone :", "ok" if r.returncode == 0 else r.stderr[:200])
        except Exception as e:
            print("clone skipped:", type(e).__name__)
    else:
        print("clone : already present")
    if REPO_DIR.exists():
        for k, v in [("user.name", CONFIG["GIT_USER_NAME"]),
                     ("user.email", CONFIG["GIT_USER_EMAIL"])]:
            subprocess.run(["git","-C",str(REPO_DIR),"config",k,v], check=False)
        subprocess.run(["git","-C",str(REPO_DIR),"checkout","-q",
                        CONFIG["GITHUB_BRANCH"]], check=False, capture_output=True)
        subprocess.run(["git","-C",str(REPO_DIR),"pull","-q","origin",
                        CONFIG["GITHUB_BRANCH"]], check=False, capture_output=True)
    ROOT = REPO_DIR if REPO_DIR.exists() else Path("/content")
else:
    ROOT = Path.cwd()
    while not (ROOT/".git").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT/".git").exists():
        ROOT = Path.cwd()

DIRS = {"dataset": ROOT/"data"/"dataset", "stats": ROOT/"data"/"stats",
        "results": ROOT/"data"/"results", "src": ROOT/"src"/"clinicalshield"}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)
print("root  :", ROOT)

for pkg, mod in [("transformers","transformers"), ("torch","torch"),
                 ("scikit-learn","sklearn")]:
    try:
        __import__(mod)
    except ImportError:
        subprocess.run([sys.executable,"-m","pip","install","-q",pkg], check=False)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

def read_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

REQUIRED_FIELDS = {"split", "template_id"}

def _fields_of(path):
    with open(path, encoding="utf-8") as f:
        return set(json.loads(f.readline()).keys())

def restore_fresh(rel_path, label):
    candidates = []
    local = ROOT / rel_path
    if local.exists():
        candidates.append((local, "repo"))
    if DRIVE_ROOT and (DRIVE_ROOT / rel_path).exists():
        candidates.append((DRIVE_ROOT / rel_path, "drive"))
    if not candidates:
        raise FileNotFoundError("\n%s not found in repo or Drive. Run NB02/NB03." % label)
    for path, src in candidates:
        missing = REQUIRED_FIELDS - _fields_of(path)
        if not missing:
            if src == "drive":
                local.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(path, local)
            return local, src
        print("  stale copy ignored (%s): missing %s" % (src, sorted(missing)))
    raise RuntimeError("\nEvery copy of %s predates the template-disjoint fix." % label)

p_ds, src_ds = restore_fresh("data/dataset/attack_dataset_split.jsonl",
                             "attack_dataset_split.jsonl")
dataset = read_jsonl(p_ds)

print("")
print("dataset from %s  %d chunks" % (src_ds, len(dataset)))
for s in ["train","val","test"]:
    rows = [d for d in dataset if d["split"] == s]
    print("  %-6s %5d chunks  %4d adversarial"
          % (s, len(rows), sum(d["label"] for d in rows)))


## Constructing the queries

Module 3 needs a query to compare against. The dataset does not carry one, so it is derived
from the host document's drug and label section — mirroring how a medication-review or
guideline-summarisation request would be phrased against that corpus.

Derived from the **host**, never the payload. This is the same discipline as the length
matching in NB02: the input must not encode the answer.

In [ ]:
# ── Cell 3 · Queries paired with retrieved documents ──────────────────────────
# Module 3 asks whether a retrieved document matches the clinical query. That
# requires a query, which the dataset does not carry -- so one is constructed per
# chunk from its own drug and label section, mirroring how a medication-review or
# guideline-summarisation query would be phrased.
#
# The query is derived from the HOST document's metadata, never from the payload.
# A query built from injected content would leak the label into the input.

SECTION_QUERY = {
    "indications_and_usage":       "What is {drug} indicated for?",
    "dosage_and_administration":   "What is the recommended dosing for {drug}?",
    "contraindications":           "What are the contraindications for {drug}?",
    "warnings_and_cautions":       "What warnings apply to {drug}?",
    "drug_interactions":           "What drug interactions are known for {drug}?",
    "boxed_warning":               "Does {drug} carry a boxed warning?",
    "adverse_reactions":           "What adverse reactions are reported for {drug}?",
    "use_in_specific_populations": "Is {drug} safe in pregnancy or renal impairment?",
}

for d in dataset:
    tpl = SECTION_QUERY.get(d["section"], "What clinical guidance applies to {drug}?")
    d["query"] = tpl.format(drug=d["drug"])

parts = {s: [d for d in dataset if d["split"] == s] for s in ["train", "val", "test"]}

print("queries built from host metadata only (drug + section), never from payload")
print("")
from collections import Counter as _C
for sec, n in sorted(_C(d["section"] for d in dataset).items()):
    print("  %-30s %5d  ->  %s"
          % (sec, n, SECTION_QUERY.get(sec, "(generic)").format(drug="X")[:44]))


## Embeddings

Mean-pooled PubMedBERT token embeddings, masked so padding does not dilute the vector, then
L2-normalised so the dot product is cosine similarity.

The AUC printed at the end is the headline diagnostic: how well similarity **alone**
separates benign from adversarial. If it sits near 0.50, embedding distance carries no signal
and the module rests entirely on its marker list.

In [ ]:
# ── Cell 4 · Embeddings and cosine similarity ─────────────────────────────────
from transformers import AutoTokenizer, AutoModel

tok = AutoTokenizer.from_pretrained(CONFIG["EMBED_MODEL"])
emb_model = AutoModel.from_pretrained(CONFIG["EMBED_MODEL"]).to(DEVICE).eval()

@torch.no_grad()
def mean_pooled(texts, batch=None):
    """Mean-pooled token embeddings, masked so padding does not dilute the vector."""
    batch = batch or CONFIG["BATCH_SIZE"]
    out = []
    for i in range(0, len(texts), batch):
        enc = tok(texts[i:i+batch], truncation=True, padding=True,
                  max_length=CONFIG["MAX_LEN"], return_tensors="pt").to(DEVICE)
        h = emb_model(**enc).last_hidden_state
        m = enc["attention_mask"].unsqueeze(-1).float()
        v = (h * m).sum(1) / m.sum(1).clamp(min=1e-9)
        out.append(torch.nn.functional.normalize(v, dim=-1).cpu())
    return torch.cat(out)

t0 = time.time()
doc_vecs = mean_pooled([d["text"] for d in dataset])
qry_vecs = mean_pooled([d["query"] for d in dataset])
cos = (doc_vecs * qry_vecs).sum(-1).numpy()
for d, c in zip(dataset, cos):
    d["cosine"] = float(c)
print("embedded %d document-query pairs in %.1f min" % (len(dataset), (time.time()-t0)/60))

ben = np.array([d["cosine"] for d in dataset if d["label"] == 0])
adv = np.array([d["cosine"] for d in dataset if d["label"] == 1])
print("")
print("cosine similarity, query vs document")
print("  benign      mean %.4f  sd %.4f  [%.3f, %.3f]"
      % (ben.mean(), ben.std(), np.percentile(ben,5), np.percentile(ben,95)))
print("  adversarial mean %.4f  sd %.4f  [%.3f, %.3f]"
      % (adv.mean(), adv.std(), np.percentile(adv,5), np.percentile(adv,95)))
print("  separation  %.4f" % abs(ben.mean() - adv.mean()))

from sklearn.metrics import roc_auc_score
y = np.array([d["label"] for d in dataset])
auc_sim = roc_auc_score(y, -np.array([d["cosine"] for d in dataset]))
print("  AUC using similarity alone : %.4f  (0.50 = no signal)" % auc_sim)


## Sentence-level scoring

Document-level pooling failed: separation 0.0008 and an AUC below chance. The cause is
dilution. A payload of roughly sixteen words inside a 250-word chunk is about 6% of the
tokens, and the remaining 94% is genuine label text about the drug the query asks about —
so the pooled vector barely moves.

This is the third appearance of one pattern. Module 1's canonicalisation drift was measured
per document and missed local changes until measured per segment. Cipher recovery failed on
whole text and worked per segment. Here a payload sentence is measured inside a document
that dwarfs it.

Measured alone, that sentence is 100% of its own vector rather than 6% of the document's.

In [ ]:
# ── Cell 4b · Sentence-level scoring ──────────────────────────────────────────
# Document-level pooling failed: separation 0.0008, AUC below chance. The reason
# is dilution. A ~16-word payload inside a ~250-word chunk is roughly 6% of the
# tokens, and the other 94% remain genuine label text about the drug the query
# asks about, so the mean-pooled vector barely moves.
#
# This is the same failure mode found twice in Module 1: a local signal measured
# globally. Measuring per sentence puts the payload on its own, where it is 100%
# of its own vector rather than 6% of the document's.

SENT_RE = re.compile(r"(?<=[.!?])\s+(?=[A-Z])|\n+")

def sentences(text, min_words=4):
    parts = [s.strip() for s in SENT_RE.split(text) if s.strip()]
    return [s for s in parts if len(s.split()) >= min_words] or [text]

# Build the sentence inventory once, remembering which chunk each came from.
sent_texts, sent_owner = [], []
for i, d in enumerate(dataset):
    for s in sentences(d["text"]):
        sent_texts.append(s)
        sent_owner.append(i)

print("sentences: %d across %d chunks (%.1f per chunk)"
      % (len(sent_texts), len(dataset), len(sent_texts)/len(dataset)))

t0 = time.time()
sent_vecs = mean_pooled(sent_texts)
print("embedded in %.1f min" % ((time.time()-t0)/60))

# Each sentence compared against its own chunk's query.
owner_qry = qry_vecs[torch.tensor(sent_owner)]
sent_cos = (sent_vecs * owner_qry).sum(-1).numpy()

# Per chunk: the least on-topic sentence it contains.
min_cos = np.full(len(dataset), 1.0)
min_sent = [""] * len(dataset)
for s_i, d_i in enumerate(sent_owner):
    if sent_cos[s_i] < min_cos[d_i]:
        min_cos[d_i] = sent_cos[s_i]
        min_sent[d_i] = sent_texts[s_i]

for d, m, s in zip(dataset, min_cos, min_sent):
    d["min_sent_cosine"] = float(m)
    d["min_sent_text"] = s

parts = {s: [d for d in dataset if d["split"] == s] for s in ["train", "val", "test"]}

ben_s = np.array([d["min_sent_cosine"] for d in dataset if d["label"] == 0])
adv_s = np.array([d["min_sent_cosine"] for d in dataset if d["label"] == 1])
auc_sent = roc_auc_score(np.array([d["label"] for d in dataset]),
                         -np.array([d["min_sent_cosine"] for d in dataset]))

print("")
print("minimum sentence similarity per chunk")
print("  benign      %.4f +/- %.4f" % (ben_s.mean(), ben_s.std()))
print("  adversarial %.4f +/- %.4f" % (adv_s.mean(), adv_s.std()))
print("  separation  %.4f   (document level was %.4f)"
      % (abs(ben_s.mean()-adv_s.mean()), abs(ben.mean()-adv.mean())))
print("  AUC         %.4f   (document level was %.4f)" % (auc_sent, auc_sim))

# split by vector: the prediction is that off-topic detection helps override
# attacks and not misinformation, because a misinformation sentence is on topic
for vec in ["override", "misinformation"]:
    v = np.array([d["min_sent_cosine"] for d in dataset
                  if d["label"] == 1 and d["vector"] == vec])
    if len(v):
        print("  %-16s %.4f +/- %.4f" % (vec, v.mean(), v.std()))


## Contradiction against the authentic label

Topic divergence cannot catch a sentence that is on topic but false. A misinformation
payload about warfarin dosing sits inside a warfarin dosing passage, answering a warfarin
dosing query. It is correctly on topic, and wrong.

The authentic openFDA text is still available, so the question changes from *does this
sentence belong here* to **does this sentence contradict its source**. Natural Language
Inference answers exactly that: given a premise and a hypothesis it returns entailment,
contradiction or neutral.

This is a different mechanism from Module 5. Module 5 had to identify a drug name, match a
claim pattern, then query an API — three brittle steps, which is why three rephrased claims
reduced it to extracting nothing. NLI compares meaning directly, so rewording does not
defeat it.

The boundary remains the same one Module 5 found: openFDA covers drug properties, not
whether a procedure is appropriate. Claims of the second kind have no machine-checkable
source, which is why MPIB's V2 attacks defeated Module 5 and will defeat this too.

In [ ]:
# ── Cell 4c · Contradiction against the authentic label ───────────────────────
# Topic divergence cannot catch a sentence that is on topic but false. A
# misinformation payload about warfarin dosing sits inside a warfarin dosing
# passage, answering a warfarin dosing query -- correctly on topic, and wrong.
#
# The authentic label text is still available, so the question becomes whether a
# sentence CONTRADICTS its source rather than whether it belongs there. Natural
# Language Inference answers exactly that: given premise and hypothesis, it
# returns entailment, contradiction or neutral.
#
# Premise: the authentic chunk text as retrieved from openFDA in NB01.
# Hypothesis: each sentence of the possibly-poisoned document.

NLI_MODEL = "microsoft/deberta-large-mnli"
RUN_NLI = True
NLI_MAX_CHUNKS = 701        # full test partition; reduce if runtime is a problem

nli_results = {}
if RUN_NLI:
    try:
        from transformers import AutoTokenizer as _AT, AutoModelForSequenceClassification as _AM
        nli_tok = _AT.from_pretrained(NLI_MODEL)
        nli_model = _AM.from_pretrained(NLI_MODEL).to(DEVICE).eval()
        LABELS = nli_model.config.id2label
        print("NLI model:", NLI_MODEL, "| labels:", LABELS)
    except Exception as e:
        print("NLI unavailable (%s); skipping" % type(e).__name__)
        RUN_NLI = False

if RUN_NLI:
    # The premise must be the authentic text the payload was injected into.
    # NB02 recorded span_start/span_end, so deleting that span reconstructs the
    # exact pre-injection host. Using a different clean chunk from the same
    # document is close but not the same passage, which weakens the comparison.

    def authentic_host(d):
        if d["label"] == 0:
            return d["text"]
        s0, e0 = d.get("span_start"), d.get("span_end")
        if s0 is None or e0 is None:
            return None
        host = (d["text"][:s0] + d["text"][e0:]).strip()
        return re.sub(r"\s+", " ", host)

    _ok = sum(1 for d in dataset if d["label"] == 1 and authentic_host(d))
    print("pre-injection host reconstructable for %d/%d adversarial chunks"
          % (_ok, sum(d["label"] for d in dataset)))

    @torch.no_grad()
    def contradiction_score(premise, hypotheses, batch=8):
        """Max contradiction probability across hypotheses."""
        best = 0.0
        worst_h = ""
        for i in range(0, len(hypotheses), batch):
            hs = hypotheses[i:i+batch]
            enc = nli_tok([premise]*len(hs), hs, truncation=True, padding=True,
                          max_length=256, return_tensors="pt").to(DEVICE)
            probs = torch.softmax(nli_model(**enc).logits, dim=-1).cpu().numpy()
            ci = [k for k, v in LABELS.items() if "CONTRADICT" in v.upper()]
            if not ci:
                return 0.0, ""
            c = probs[:, ci[0]]
            j = int(c.argmax())
            if c[j] > best:
                best, worst_h = float(c[j]), hs[j]
        return best, worst_h

    # Scored on validation AND test: the threshold is selected on validation in
    # Cell 5, so scoring test alone would leave nothing to select from.
    rng_nli = random.Random(SEED)
    sample = []
    for part in ["val", "test"]:
        rows = [d for d in parts[part] if authentic_host(d)]
        rng_nli.shuffle(rows)
        sample += rows[:NLI_MAX_CHUNKS]

    print("scoring %d chunks (%d val, %d test)"
          % (len(sample),
             sum(1 for d in sample if d["split"] == "val"),
             sum(1 for d in sample if d["split"] == "test")))

    t0 = time.time()
    for k, d in enumerate(sample):
        premise = authentic_host(d)[:1200]
        hyps = sentences(d["text"])[:12]
        score_c, worst = contradiction_score(premise, hyps)
        d["contradiction"] = score_c
        d["contradiction_sentence"] = worst
        if (k+1) % 200 == 0:
            print("  %d/%d  %.1f min" % (k+1, len(sample), (time.time()-t0)/60))

    scored = [d for d in sample if d["split"] == "test" and "contradiction" in d]
    cb = np.array([d["contradiction"] for d in scored if d["label"] == 0])
    ca = np.array([d["contradiction"] for d in scored if d["label"] == 1])
    auc_nli = (roc_auc_score([d["label"] for d in scored],
                             [d["contradiction"] for d in scored])
               if len({d["label"] for d in scored}) > 1 else float("nan"))

    print("")
    print("CONTRADICTION AGAINST AUTHENTIC LABEL (n=%d sampled test chunks)" % len(scored))
    print("  benign      %.4f +/- %.4f  (n=%d)" % (cb.mean(), cb.std(), len(cb)))
    print("  adversarial %.4f +/- %.4f  (n=%d)" % (ca.mean(), ca.std(), len(ca)))
    print("  separation  %.4f" % abs(cb.mean()-ca.mean()))
    print("  AUC         %.4f" % auc_nli)

    for vec in ["override", "misinformation"]:
        v = np.array([d["contradiction"] for d in scored
                      if d["label"] == 1 and d["vector"] == vec])
        if len(v):
            print("  %-16s %.4f +/- %.4f  (n=%d)" % (vec, v.mean(), v.std(), len(v)))

    nli_results = {
        "model": NLI_MODEL, "n_scored": len(scored),
        "benign_mean": float(cb.mean()), "adversarial_mean": float(ca.mean()),
        "auc": float(auc_nli),
        "by_vector": {vec: float(np.mean([d["contradiction"] for d in scored
                                          if d["label"] == 1 and d["vector"] == vec]))
                      for vec in ["override", "misinformation"]
                      if any(d["label"] == 1 and d["vector"] == vec for d in scored)},
    }

    del nli_model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


## Choosing the threshold on validation

Two sweeps: similarity alone, then similarity combined with the markers. Both on validation.

Running the sweep on test and then reporting the best point would be selecting the operating
point to flatter the result — a subtler version of the error Reviewer 4 caught in v1.

In [ ]:
# ── Cell 5 · Threshold selection on VALIDATION ────────────────────────────────
# v1 used tau = 0.30 with no stated justification. Here the sweep runs on the
# validation partition, the operating point is fixed, and test is evaluated once
# at that point -- so the reported number is not chosen to flatter itself.

def flag(d, tau, use_markers=True, level="document", tau_contra=None):
    """Module 3 raises a flag on any of three signals.

    similarity    the document (or its least on-topic sentence) diverges from
                  the query in meaning
    marker        it carries a known instruction-override phrase
    contradiction it contradicts the authentic source passage

    The third exists because the first two are blind to a sentence that is on
    topic and false. A misinformation payload about warfarin dosing sits inside a
    warfarin dosing passage answering a warfarin dosing query: correctly on
    topic, and wrong. Measured on validation, similarity separated benign from
    adversarial at AUC 0.56 and caught no misinformation at all, while
    contradiction against the authentic label reached AUC 0.71 and produced a
    signal on misinformation for the first time.
    """
    key = "cosine" if level == "document" else "min_sent_cosine"
    if d.get(key, 1.0) < tau:
        return True, "similarity"
    if use_markers:
        low = d["text"].lower()
        for m in CONFIG["OVERRIDE_MARKERS"]:
            if m in low:
                return True, "marker"
    if tau_contra is not None and d.get("contradiction") is not None:
        if d["contradiction"] >= tau_contra:
            return True, "contradiction"
    return False, None

def score(rows, tau, use_markers=True, level="document", tau_contra=None):
    tp = fp = fn = tn = 0
    by_reason = Counter()
    for d in rows:
        f, why = flag(d, tau, use_markers, level, tau_contra)
        if f:
            by_reason[why] += 1
        if d["label"] == 1:
            tp += f; fn += (not f)
        else:
            fp += f; tn += (not f)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    return {"tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "precision": prec, "recall": rec,
            "f1": 2*prec*rec/(prec+rec) if (prec+rec) else 0.0,
            "fpr": fp/(fp+tn) if (fp+tn) else 0.0,
            "reasons": dict(by_reason)}

print("VALIDATION — DOCUMENT-level similarity only (v1 design)")
print("  tau     recall    FPR      F1")
sim_only = {}
for tau in CONFIG["TAU_SWEEP"]:
    s = score(parts["val"], tau, use_markers=False, level="document")
    sim_only[tau] = s
    print("  %.2f   %.4f   %.4f   %.4f" % (tau, s["recall"], s["fpr"], s["f1"]))

print("")
print("VALIDATION — SENTENCE-level similarity only (v2)")
print("  tau     recall    FPR      F1")
sent_only = {}
for tau in CONFIG["TAU_SWEEP"]:
    s = score(parts["val"], tau, use_markers=False, level="sentence")
    sent_only[tau] = s
    print("  %.2f   %.4f   %.4f   %.4f" % (tau, s["recall"], s["fpr"], s["f1"]))

print("")
print("VALIDATION — sentence similarity + override markers")
print("  tau     recall    FPR      F1")
combined = {}
for tau in CONFIG["TAU_SWEEP"]:
    s = score(parts["val"], tau, use_markers=True, level="sentence")
    combined[tau] = s
    print("  %.2f   %.4f   %.4f   %.4f" % (tau, s["recall"], s["fpr"], s["f1"]))

# ── operating point selection ────────────────────────────────────────────────
# Selecting by F1 is wrong here. Benign and adversarial cosines overlap heavily,
# so F1 is maximised by driving the threshold up and flagging almost everything:
# in development that produced tau = 0.90 at a 79% false positive rate. A CDSS
# that flags four documents in five is not deployable, and the paper's own
# argument is that false positives matter more than detection rate.
#
# The threshold is therefore chosen to maximise recall subject to an explicit
# false-positive budget, stated in advance rather than discovered afterwards.

FPR_BUDGET = 0.05

feasible = {t: s for t, s in combined.items() if s["fpr"] <= FPR_BUDGET}
if feasible:
    best_tau = max(feasible, key=lambda t: feasible[t]["recall"])
    basis = "max recall subject to FPR <= %.0f%%" % (100 * FPR_BUDGET)
else:
    best_tau = min(combined, key=lambda t: combined[t]["fpr"])
    basis = "no threshold met the FPR budget; lowest-FPR point selected"

print("")
print("OPERATING POINT")
print("  criterion       : %s" % basis)
print("  FPR budget      : %.0f%%" % (100 * FPR_BUDGET))
print("  thresholds meeting budget : %d of %d" % (len(feasible), len(combined)))
print("  selected tau    : %.2f   (v1 used %.2f)" % (best_tau, CONFIG["TAU_INTENT"]))
s_sel = combined[best_tau]
print("  validation      : recall %.4f  FPR %.4f  precision %.4f"
      % (s_sel["recall"], s_sel["fpr"], s_sel["precision"]))

# ── contradiction threshold, selected on validation under the same budget ────
TAU_CONTRA = None
if nli_results:
    scored_val = [d for d in parts["val"] if d.get("contradiction") is not None]
    if len(scored_val) >= 50:
        print("")
        print("VALIDATION — contradiction threshold sweep (n=%d scored)" % len(scored_val))
        print("  tau_c   recall    FPR      precision")
        contra_sweep = {}
        for tc in [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]:
            sc = score(scored_val, best_tau, use_markers=True,
                       level="sentence", tau_contra=tc)
            contra_sweep[tc] = sc
            print("  %.2f   %.4f   %.4f   %.4f"
                  % (tc, sc["recall"], sc["fpr"], sc["precision"]))
        feas_c = {t: v for t, v in contra_sweep.items() if v["fpr"] <= FPR_BUDGET}
        if feas_c:
            TAU_CONTRA = max(feas_c, key=lambda t: feas_c[t]["recall"])
            print("")
            print("  selected tau_contra %.2f  (recall %.4f, FPR %.4f)"
                  % (TAU_CONTRA, feas_c[TAU_CONTRA]["recall"], feas_c[TAU_CONTRA]["fpr"]))
        else:
            print("  no contradiction threshold met the %.0f%% FPR budget"
                  % (100*FPR_BUDGET))
    else:
        print("")
        print("contradiction not scored on validation; run NLI over val to enable")

f1_tau = max(combined, key=lambda t: combined[t]["f1"])
print("")
print("  for contrast, the F1-optimal point would be tau %.2f at FPR %.4f"
      % (f1_tau, combined[f1_tau]["fpr"]))


## Held-out evaluation, with the signals separated

Three configurations at the validation-selected threshold: similarity only, markers only, and
both.

Reporting them separately is the point. If markers-only matches both-combined, then the
similarity check contributes nothing and the paper should say so rather than describe a
two-signal module that is effectively one.

In [ ]:
# ── Cell 6 · Held-out evaluation, and what each signal contributes ────────────
# Evaluated once, at the validation-selected operating point.

TAU = best_tau
test_scored = [d for d in parts["test"] if d.get("contradiction") is not None]

test_combined    = score(parts["test"], TAU, use_markers=True,  level="sentence")
test_sent_only   = score(parts["test"], TAU, use_markers=False, level="sentence")
test_sim_only    = score(parts["test"], TAU, use_markers=False, level="document")
test_marker_only = score(parts["test"], -1.0, use_markers=True)   # tau unreachable

print("HELD-OUT TEST at tau = %.2f (selected on validation)" % TAU)
print("")
print("configuration          recall     FPR       precision   F1")
for name, s in [("document similarity", test_sim_only),
                ("sentence similarity", test_sent_only),
                ("markers only", test_marker_only),
                ("sentence + markers", test_combined)]:
    print("  %-20s %.4f   %.4f    %.4f     %.4f"
          % (name, s["recall"], s["fpr"], s["precision"], s["f1"]))

# three-signal configuration, on the subset where contradiction was scored
if TAU_CONTRA is not None and test_scored:
    base3  = score(test_scored, TAU, use_markers=True, level="sentence")
    full3  = score(test_scored, TAU, use_markers=True, level="sentence",
                   tau_contra=TAU_CONTRA)
    print("")
    print("THREE SIGNALS, on the %d test chunks scored by NLI (tau_c = %.2f)"
          % (len(test_scored), TAU_CONTRA))
    print("  configuration                recall     FPR       precision")
    print("  sentence + markers           %.4f   %.4f    %.4f"
          % (base3["recall"], base3["fpr"], base3["precision"]))
    print("  + contradiction              %.4f   %.4f    %.4f"
          % (full3["recall"], full3["fpr"], full3["precision"]))
    print("  gain from contradiction      %+.4f recall, %+.4f FPR"
          % (full3["recall"]-base3["recall"], full3["fpr"]-base3["fpr"]))
    print("")
    print("  recall by vector, three signals")
    for vec in ["override", "misinformation"]:
        rows = [d for d in test_scored if d["label"] == 1 and d["vector"] == vec]
        if not rows:
            continue
        b = sum(flag(d, TAU, True, "sentence")[0] for d in rows) / len(rows)
        f = sum(flag(d, TAU, True, "sentence", TAU_CONTRA)[0] for d in rows) / len(rows)
        print("    %-16s n=%4d   two signals %.4f   three signals %.4f   %+.4f"
              % (vec, len(rows), b, f, f-b))
    print("")
    print("  signals fired : %s" % full3["reasons"])
else:
    full3 = None
    print("")
    print("contradiction signal not enabled (no threshold met the FPR budget)")

print("")
print("which signal fired, among detections (two-signal config)")
print("  ", test_combined["reasons"])

# Recall by vector: the finding v1 reported on MPIB, reproduced on own data
print("")
print("recall by attack vector, Module 3 alone")
for vec in ["override", "misinformation"]:
    rows = [d for d in parts["test"] if d["label"] == 1 and d["vector"] == vec]
    if not rows:
        continue
    hit = sum(flag(d, TAU, True, "sentence")[0] for d in rows)
    print("  %-16s n=%4d  recall %.4f" % (vec, len(rows), hit/len(rows)))

print("")
print("recall by category, Module 3 alone")
for cat in sorted({d["category"] for d in parts["test"] if d["label"] == 1}):
    rows = [d for d in parts["test"] if d["label"] == 1 and d["category"] == cat]
    hit = sum(flag(d, TAU, True, "sentence")[0] for d in rows)
    print("  %-28s n=%4d  recall %.4f" % (cat, len(rows), hit/len(rows)))


## Export and persist

In [ ]:
# ── Cell 7 · Export Module 3 and save results ─────────────────────────────────
MODULE3_SRC = '''"""ClinicalShield Module 3 — semantic intent verification.

Flags a retrieved document when it diverges from the clinical query in meaning,
or when it carries a known instruction-override marker. Generated by
notebooks/06_module3_intent.ipynb.
"""

import torch
import torch.nn.functional as F

OVERRIDE_MARKERS = %s

TAU_INTENT = %.2f
TAU_CONTRADICTION = TAU_CONTRA_PLACEHOLDER


def mean_pooled(texts, tokenizer, model, device="cpu", max_len=512, batch=32):
    out = []
    with torch.no_grad():
        for i in range(0, len(texts), batch):
            enc = tokenizer(texts[i:i+batch], truncation=True, padding=True,
                            max_length=max_len, return_tensors="pt").to(device)
            h = model(**enc).last_hidden_state
            m = enc["attention_mask"].unsqueeze(-1).float()
            v = (h * m).sum(1) / m.sum(1).clamp(min=1e-9)
            out.append(F.normalize(v, dim=-1).cpu())
    return torch.cat(out)


def module3(document, query, cosine=None, contradiction=None,
            tau=TAU_INTENT, tau_contradiction=TAU_CONTRADICTION):
    """Semantic intent verification over three signals.

    cosine        minimum per-sentence similarity to the query. Document-level
                  pooling dilutes a short payload to roughly 6%% of the vector
                  and does not separate the classes.
    markers       literal instruction-override phrases.
    contradiction probability that some sentence contradicts the authentic
                  source passage. This is the only one of the three that
                  produces a signal on misinformation payloads, which are on
                  topic by construction and therefore invisible to similarity.

    Returns (flagged, reason).
    """
    if cosine is not None and cosine < tau:
        return True, "similarity"
    low = document.lower()
    for m in OVERRIDE_MARKERS:
        if m in low:
            return True, "marker"
    if (contradiction is not None and tau_contradiction is not None
            and contradiction >= tau_contradiction):
        return True, "contradiction"
    return False, None
''' % (repr(CONFIG["OVERRIDE_MARKERS"]), TAU)

MODULE3_SRC = MODULE3_SRC.replace(
    "TAU_CONTRA_PLACEHOLDER",
    ("%.2f" % TAU_CONTRA) if TAU_CONTRA is not None else "None")
(DIRS["src"] / "module3.py").write_text(MODULE3_SRC)
print("exported src/clinicalshield/module3.py")

module3_results = {
    "notebook": "06_module3_intent",
    "seed": SEED,
    "embed_model": CONFIG["EMBED_MODEL"],
    "protocol": {
        "tau_selected_on": ("validation partition; maximum recall subject to a "
                            "false-positive budget of 5%"),
        "fpr_budget": FPR_BUDGET,
        "tau_selected": float(TAU),
        "tau_v1": CONFIG["TAU_INTENT"],
        "note": ("v1 reported tau = 0.30 without stated justification. The sweep "
                 "runs on validation and the operating point is applied unchanged "
                 "to test. Selection is by recall under an explicit false-positive "
                 "budget rather than by F1: benign and adversarial cosines overlap "
                 "heavily, so F1 is maximised at a threshold that flags most "
                 "documents, which is not a deployable operating point for a CDSS."),
        "query_construction": ("derived from host document metadata (drug and label "
                               "section) only; never from injected payload text"),
        "n_override_markers": len(CONFIG["OVERRIDE_MARKERS"]),
        "scoring_level": "sentence (minimum similarity); document level reported for contrast",
    },
    "similarity_distribution": {
        "document_level": {
            "benign_mean": float(ben.mean()), "benign_sd": float(ben.std()),
            "adversarial_mean": float(adv.mean()), "adversarial_sd": float(adv.std()),
            "auc": float(auc_sim),
            "note": ("v1 design. A ~16-word payload in a ~250-word chunk is ~6% of "
                     "tokens, so mean pooling leaves the vector essentially unmoved."),
        },
        "sentence_level": {
            "benign_mean": float(ben_s.mean()), "benign_sd": float(ben_s.std()),
            "adversarial_mean": float(adv_s.mean()), "adversarial_sd": float(adv_s.std()),
            "auc": float(auc_sent),
            "note": "minimum per-sentence similarity to the query",
        },
    },
    "contradiction_nli": nli_results,
    "validation_sweep": {
        "similarity_only": {str(k): {kk: vv for kk, vv in v.items() if kk != "reasons"}
                            for k, v in sim_only.items()},
        "combined": {str(k): {kk: vv for kk, vv in v.items() if kk != "reasons"}
                     for k, v in combined.items()},
    },
    "three_signal": ({
        "tau_contradiction": float(TAU_CONTRA),
        "n_scored_test": len(test_scored),
        "two_signal": {k: v for k, v in base3.items() if k != "reasons"},
        "three_signal": {k: v for k, v in full3.items() if k != "reasons"},
        "reasons": full3["reasons"],
        "by_vector": {
            vec: {
                "n": len([d for d in test_scored
                          if d["label"] == 1 and d["vector"] == vec]),
                "two_signal": (sum(flag(d, TAU, True, "sentence")[0]
                                   for d in test_scored
                                   if d["label"] == 1 and d["vector"] == vec)
                               / max(1, len([d for d in test_scored
                                             if d["label"] == 1 and d["vector"] == vec]))),
                "three_signal": (sum(flag(d, TAU, True, "sentence", TAU_CONTRA)[0]
                                     for d in test_scored
                                     if d["label"] == 1 and d["vector"] == vec)
                                 / max(1, len([d for d in test_scored
                                               if d["label"] == 1 and d["vector"] == vec]))),
            } for vec in ["override", "misinformation"]
        },
    } if (TAU_CONTRA is not None and full3) else None),
    "test": {
        "tau": float(TAU),
        "document_similarity_only": test_sim_only,
        "sentence_similarity_only": test_sent_only,
        "similarity_only": test_sent_only,
        "markers_only": test_marker_only,
        "combined": test_combined,
    },
    "generated_at": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
}
with open(DIRS["results"] / "module3_results.json", "w") as f:
    json.dump(module3_results, f, indent=2)

if IN_COLAB and DRIVE_ROOT:
    dst = DRIVE_ROOT / "data" / "results"; dst.mkdir(parents=True, exist_ok=True)
    for f_ in (ROOT / "data" / "results").glob("*"):
        shutil.copy2(f_, dst / f_.name)
    print("mirrored to Drive")

if IN_COLAB and CONFIG["PUSH_TO_GITHUB"] and REPO_DIR and REPO_DIR.exists():
    keep = ["data/results/module3_results.json", "src/clinicalshield/module3.py"]
    subprocess.run(["git","-C",str(ROOT),"add","-f"] + keep, check=False)
    st = subprocess.run(["git","-C",str(ROOT),"status","--porcelain"],
                        capture_output=True, text=True)
    if st.stdout.strip():
        subprocess.run(["git","-C",str(ROOT),"commit","-q","-m",
                        "NB06: Module 3, tau=%.2f selected on validation, "
                        "test recall %.4f" % (TAU, test_combined["recall"])], check=False)
        pr = subprocess.run(["git","-C",str(ROOT),"push","-q","origin",
                             CONFIG["GITHUB_BRANCH"]], capture_output=True, text=True)
        print("push:", "ok" if pr.returncode == 0 else pr.stderr[:200])

print("written: module3_results.json")


## Summary

**Paste the output below into the chat.**

The two numbers I will look at first: the AUC on similarity alone, and whether markers-only
differs from both-combined. Between them they determine whether Module 3 is one mechanism or
two.

In [ ]:
# ── Cell 8 · NB06 SUMMARY — paste this output ─────────────────────────────────
print("=" * 68)
print("NB06 — MODULE 3: SEMANTIC INTENT VERIFICATION")
print("=" * 68)
print("embed model : %s" % CONFIG["EMBED_MODEL"].split("/")[-1])
print("tau         : %.2f  (validation, max recall at FPR <= %.0f%%; v1 used %.2f)"
      % (TAU, 100*FPR_BUDGET, CONFIG["TAU_INTENT"]))
print("markers     : %d override phrases" % len(CONFIG["OVERRIDE_MARKERS"]))
print("queries     : built from host metadata only, never from payload")
print("")
print("SIMILARITY — document level (v1) vs sentence level (v2)")
print("  level       benign           adversarial      separation   AUC")
print("  document    %.4f+/-%.4f   %.4f+/-%.4f   %.4f      %.4f"
      % (ben.mean(), ben.std(), adv.mean(), adv.std(),
         abs(ben.mean()-adv.mean()), auc_sim))
print("  sentence    %.4f+/-%.4f   %.4f+/-%.4f   %.4f      %.4f"
      % (ben_s.mean(), ben_s.std(), adv_s.mean(), adv_s.std(),
         abs(ben_s.mean()-adv_s.mean()), auc_sent))
if nli_results:
    print("")
    print("CONTRADICTION vs AUTHENTIC LABEL (NLI, n=%d)" % nli_results["n_scored"])
    print("  benign      %.4f" % nli_results["benign_mean"])
    print("  adversarial %.4f" % nli_results["adversarial_mean"])
    print("  AUC         %.4f" % nli_results["auc"])
    for v, m in nli_results.get("by_vector", {}).items():
        print("  %-16s %.4f" % (v, m))
print("")
print("HELD-OUT TEST at tau = %.2f" % TAU)
print("configuration          recall     FPR       precision   F1")
for name, s in [("document similarity", test_sim_only),
                ("sentence similarity", test_sent_only),
                ("markers only", test_marker_only),
                ("sentence + markers", test_combined)]:
    print("  %-20s %.4f   %.4f    %.4f     %.4f"
          % (name, s["recall"], s["fpr"], s["precision"], s["f1"]))
print("")
print("detections by signal : %s" % test_combined["reasons"])
if TAU_CONTRA is not None and full3:
    print("")
    print("THREE SIGNALS (n=%d test chunks scored by NLI, tau_c=%.2f)"
          % (len(test_scored), TAU_CONTRA))
    print("  configuration            recall     FPR       precision")
    print("  sentence + markers       %.4f   %.4f    %.4f"
          % (base3["recall"], base3["fpr"], base3["precision"]))
    print("  + contradiction          %.4f   %.4f    %.4f"
          % (full3["recall"], full3["fpr"], full3["precision"]))
    print("  signals fired : %s" % full3["reasons"])
    print("")
    print("  by vector          n     two-signal   three-signal   gain")
    for vec in ["override", "misinformation"]:
        rows = [d for d in test_scored if d["label"] == 1 and d["vector"] == vec]
        if not rows:
            continue
        b = sum(flag(d, TAU, True, "sentence")[0] for d in rows)/len(rows)
        f = sum(flag(d, TAU, True, "sentence", TAU_CONTRA)[0] for d in rows)/len(rows)
        print("    %-16s %4d   %.4f       %.4f       %+.4f" % (vec, len(rows), b, f, f-b))

print("")
print("RECALL BY VECTOR (Module 3 alone)")
for vec in ["override", "misinformation"]:
    rows = [d for d in parts["test"] if d["label"] == 1 and d["vector"] == vec]
    if rows:
        hit = sum(flag(d, TAU, True, "sentence")[0] for d in rows)
        print("  %-16s n=%4d  %.4f" % (vec, len(rows), hit/len(rows)))
print("")
print("RECALL BY CATEGORY (Module 3 alone)")
for cat in sorted({d["category"] for d in parts["test"] if d["label"] == 1}):
    rows = [d for d in parts["test"] if d["label"] == 1 and d["category"] == cat]
    hit = sum(flag(d, TAU)[0] for d in rows)
    print("  %-28s n=%4d  %.4f" % (cat, len(rows), hit/len(rows)))
print("=" * 68)
print("NB06 COMPLETE — ready for NB07 (Module 4 + CEPS v2)")
print("=" * 68)
